# SmallHands: Compressing/Stress-Testing Vision-Language-Action Robot Policies

Fine-tuning, quantizing and stress-testing SmolVLA on LIBERO.

### One-time setup
1. **GPU:** *Runtime → Change runtime type →* **L4** (best value) or **A100** (fastest, uses compute units quickly). Avoid the T4: it has no native bfloat16 and is slow for training.
2. **Hugging Face token:** create a token with *write* access at huggingface.co → Settings → Access Tokens.

In Colab: click the **key icon** (Secrets) in the left sidebar → add it as `HF_TOKEN` → turn on *Notebook access*. Your checkpoints and results are backed up to a **private** Hub repo every 15 minutes.


### How to use it
- Pick what to run in the **CONFIG** cell (`STAGES`), then *Runtime → Run all*.
- **Keep the tab open** during long runs. On Colab Pro, closing the browser or leaving it idle too long ends the session.
- **If Colab disconnects:** reconnect and *Run all* again with the same `STAGES`. Setup takes about 10 minutes, your progress is restored from the Hub, and training resumes from the last checkpoint.
- **Watch your compute units** (*Runtime → View resources*). `TRAIN_HOURS` caps each training run, so one session can't burn your whole month.



## CONFIG
Training & Evaluation of B

In [ ]:
#   "tests", "baseline_smoke", "baseline", "speedtest", "train_A", "train_B",
#   "eval_trained", "quant_list", "quant", "paraphrase_generate", "paraphrase_eval", "results"
STAGES = ["train_B"]

GITHUB_URL = "https://github.com/arckived/vla-study.git"
PROJECT_ZIP = ""                 #nit used

#ugging Face backup repo (private; created automatically)
HF_RUNS_REPO = "arckive71/vla-study-runs"

# Experiment settings. identical for A and B so the comparison is fair.
SUITE = "libero_object"
N_EP = 10                        # episodes per task in full evals (10 tasks -> 100 episodes)
STEPS = 20000                    # pick after the speed test; 10000 is a fine fallback
BS = 32                        # None = pick by GPU (A100: 32, L4: 16, T4: 8). Set a number to override
SAVE_FREQ = 1000                 # frequent checkpoints = little lost if a session ends
EVAL_FREQ = 0                    # 0 = no sim eval during training (saves compute; we evaluate afterwards)
TRAIN_HOURS = 6                  # cap per training session (protects compute units); re-run to continue
EXTRA_TRAIN_ARGS = ""            # e.g. "dataset.video_backend=pyav" if video decoding errors appear

# Quantization regexes: set after running the "quant_list" stage
VLM_RE = r"\.vlm\."
EXPERT_RE = r"expert"


# Optional LLM paraphrases (needs extra GPU time); "" to skip
LLM = ""                         # e.g. "Qwen/Qwen2.5-1.5B-Instruct"

## 1. GPU check and paths

In [ ]:
import os, subprocess, sys, json, shutil, glob, re, time, signal, threading
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert gpu, "No GPU. Runtime -> Change runtime type -> L4 or A100."
if BS is None:
    BS = 32 if "A100" in gpu or "H100" in gpu else 16 if "L4" in gpu else 8
if "T4" in gpu:
    print("WARNING: T4 detected - fine for evaluation, slow for training (no native bfloat16).")
print("Batch size:", BS, "- keep the same value for experiments A and B")

WORK    = Path("/content")
PROJECT = WORK / "vla-study"                       
OUT     = WORK / "vla-outputs"                     # checkpoints, evals, results (backed up to the Hub)
VENV    = WORK / "venv"                            # Python 3.12 environment with LeRobot
LEROBOT = WORK / "lerobot"
os.environ["HF_HOME"] = "/content/hf"              # model/dataset cache
OUT.mkdir(parents=True, exist_ok=True)
want = lambda stage: stage in STAGES
COMMON = dict(SUITE=SUITE, N_EP=N_EP, STEPS=STEPS, BS=BS, SAVE_FREQ=SAVE_FREQ, EVAL_FREQ=EVAL_FREQ,
              EXTRA_TRAIN_ARGS=EXTRA_TRAIN_ARGS, DEVICE="cuda")   # passed to every script
print("Stages this session:", STAGES)

## 2. Helper for running commands

Streams the important lines, saves the full log to `vla-outputs/logs/`, and can stop a command cleanly after a time limit.

In [ ]:
def run(cmd, log=None, show=None, env=None, hours=None, ok_codes=(0,)):
    # Run a shell command
    #   show : regex - only print matching lines (None = print everything)
    #   log  : file name under vla-outputs/logs/ (full output is always kept there)
    #   hours: send Ctrl-C to the whole process group after this many hours
    e = os.environ.copy(); e.update({k: str(v) for k, v in (env or {}).items()})
    e["MPLBACKEND"] = "Agg"
    (OUT / "logs").mkdir(exist_ok=True)
    logf = open(OUT / "logs" / log, "a") if log else None
    pat = re.compile(show) if show else None
    p = subprocess.Popen(["bash", "-c", cmd], cwd=PROJECT if PROJECT.exists() else WORK, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         start_new_session=True)
    timed_out = threading.Event()
    if hours:
        def stop():
            timed_out.set(); print(f">>> time budget of {hours} h reached, stopping cleanly", flush=True)
            os.killpg(p.pid, signal.SIGINT)
        timer = threading.Timer(hours * 3600, stop); timer.daemon = True; timer.start()
    tail = []
    for line in p.stdout:
        if logf: logf.write(line); logf.flush()
        tail = (tail + [line])[-40:]
        if pat is None or pat.search(line): print(line, end="", flush=True)
    code_ = p.wait()
    if logf: logf.close()
    if timed_out.is_set():
        return "timeout"
    if code_ not in ok_codes:
        print("".join(tail))
        raise RuntimeError(f"command failed (exit {code_}): {cmd}")
    return code_

IMPORTANT = r"step|loss|success|pc_success|\[quant\]|\[paraphrase\]|\[sync\]|Error|error|Traceback|WARNING|Warning"

## 3. project code

In [ ]:
shutil.rmtree(PROJECT, ignore_errors=True)
if GITHUB_URL:
    run(f"git clone -q {GITHUB_URL} {PROJECT}")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree("/content/drive/MyDrive/vla-study", PROJECT)

# Point outputs/, data/ and results/ at vla-outputs so everything lands in one backed-up place
for name, target in [("outputs", OUT), ("data", OUT / "data"), ("results", OUT / "results")]:
    target.mkdir(parents=True, exist_ok=True)
    link = PROJECT / name
    if link.is_symlink() or link.is_file(): link.unlink()
    elif link.is_dir():
        for f in link.iterdir():                          
            if not (target / f.name).exists(): shutil.move(str(f), target / f.name)
        shutil.rmtree(link)
    link.symlink_to(target)
os.chdir(PROJECT)
print(sorted(p.name for p in PROJECT.iterdir()))

## 4. Hugging Face login and restore previous progress

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(f"Could not read the HF_TOKEN secret ({e}). Key icon in the left sidebar -> add HF_TOKEN -> enable notebook access.")
assert "<your-hf-username>" not in HF_RUNS_REPO, "Set HF_RUNS_REPO in the CONFIG cell."

# The sync tool only needs huggingface_hub, which Colab's own Python already has
def sync_cmd(action, extra=""):
    return f"{sys.executable} -m vla_study.hub_sync {action} --repo {HF_RUNS_REPO} --outputs {OUT} {extra}"
run(sync_cmd("pull"))
print("Restored files:", sorted(str(p.relative_to(OUT)) for p in OUT.glob("*")))

## 5. Install Python 3.12 environment with LeRobot

LeRobot needs Python ≥ 3.12 and a recent PyTorch, build a separate environment with uv instead of installing into Colab's own Python (which avoids version clashes and 'restart session' prompts). The LeRobot commit is pinned on the first run and re-used afterwards, so all your runs use the same code.

In [ ]:
run("pip install -q uv")
run("apt-get -qq update && apt-get -qq install -y ffmpeg libegl1 libgl1 libosmesa6 > /dev/null", ok_codes=(0, 100))

PIN = OUT / "LEROBOT_COMMIT.txt"
if not LEROBOT.exists():
    run(f"git clone -q --filter=blob:none https://github.com/huggingface/lerobot.git {LEROBOT}")
if PIN.exists():
    run(f"cd {LEROBOT} && git checkout -q {PIN.read_text().strip()}")
else:
    PIN.write_text(subprocess.check_output(["git", "-C", str(LEROBOT), "rev-parse", "HEAD"], text=True).strip())
print("LeRobot commit:", PIN.read_text().strip())

if not (VENV / "bin" / "python").exists():
    run(f"uv venv -q {VENV} --python 3.12")
run(f"uv pip install -q --python {VENV}/bin/python --torch-backend=auto "
    f"-e '{LEROBOT}[smolvla,libero]' -e {PROJECT} pytest matplotlib", show=r"error|Error")

# From now on, `python`, `lerobot-train`, `lerobot-eval` all mean the 3.12 environment
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = str(VENV)
# LIBERO may ask a yes/no question on first import; answer it so background runs don't hang
run("printf 'N\\n' | python -c 'import libero.libero' || true")
run("python -c \"import sys, torch, lerobot; print('python', sys.version.split()[0], '| torch', torch.__version__, "
    "'| CUDA', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())\"")

## 6. Check the simulator can render

The GPU renderer (EGL) is fastest. If it doesn't work on runtime, fall back to the CPU renderer (OSMesa), which is slower but works.

In [ ]:
test = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); "
        "r = mujoco.Renderer(m, 64, 64); r.update_scene(mujoco.MjData(m)); print(r.render().shape)")
for backend in ("egl", "osmesa"):
    r = subprocess.run(["python", "-c", test], capture_output=True, text=True,
                       env={**os.environ, "MUJOCO_GL": backend, "PYOPENGL_PLATFORM": backend})
    if r.returncode == 0:
        os.environ["MUJOCO_GL"] = os.environ["PYOPENGL_PLATFORM"] = backend
        print(f"Rendering works with {backend}" + ("" if backend == "egl" else " (CPU - evals will be slower)"))
        break
    print(f"{backend} failed: {r.stderr.strip().splitlines()[-1] if r.stderr.strip() else '?'}")
else:
    raise SystemExit("No working renderer; see the troubleshooting notes at the end.")

## 7. Start background backups

Every 15 minutes: delete old checkpoints (keeps the newest two per experiment) and upload new results plus the latest checkpoint to private Hub repo.

In [ ]:
subprocess.run(["pkill", "-f", "vla_study.hub_sync loop"])        # avoid duplicates if this cell re-runs
(OUT / "logs").mkdir(exist_ok=True)
sync_log = open(OUT / "logs" / "sync.log", "a")
sync_proc = subprocess.Popen(sync_cmd("loop", "--every 900 --keep 2"),
                             shell=True, cwd=PROJECT, stdout=sync_log, stderr=subprocess.STDOUT)
print("backup loop running, pid", sync_proc.pid, "- log: vla-outputs/logs/sync.log")

---
# Stages
Each cell below only runs if its stage is listed in `STAGES`.

### tests: unit tests (should all pass)

In [ ]:
if want("tests"):
    run("python -m pytest tests -q")

### baseline_smoke / baseline: check the evaluation pipeline (do NOT skip)

The smoke test runs 1 episode per task. If the full baseline is near 0% success, stop and debug using Part 4 of the learning guide before training anything.

In [ ]:
from IPython.display import Video, display

def show_video(pattern):
    vids = sorted(glob.glob(str(OUT / "eval" / pattern / "**" / "*.mp4"), recursive=True))
    print(len(vids), "videos")
    if vids: display(Video(vids[0], embed=True, width=400))

if want("baseline_smoke"):
    run("bash scripts/01_eval_baseline.sh", log="baseline_smoke.log", show=IMPORTANT, env={**COMMON, "N_EP": 1})
    show_video("00_baseline_pretrained")
if want("baseline"):
    shutil.rmtree(OUT / "eval" / "00_baseline_pretrained", ignore_errors=True)
    run("bash scripts/01_eval_baseline.sh", log="baseline.log", show=IMPORTANT, env=COMMON)
    show_video("00_baseline_pretrained")

### speedtest: how long will training take?

Trains 200 steps, then deletes the run. Read the seconds-per-step from the log: total hours ≈ sec/step × STEPS ÷ 3600. Multiply the hours by your GPU's units per hour (*View resources*) to see the cost. If 20k steps won't fit, use 10k, and use the same value for A and B.

In [ ]:
if want("speedtest"):
    run("bash scripts/02_train_expert_only.sh", log="speedtest.log", show=IMPORTANT,
        env={**COMMON, "STEPS": 200, "SAVE_FREQ": 1000, "EVAL_FREQ": 0})
    shutil.rmtree(OUT / "train" / "expert_only", ignore_errors=True)
    run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

### train_A / train_B: training

Starts fresh the first time, and automatically resumes from the last checkpoint after that, including after a disconnect. Stops itself after TRAIN_HOURS; run again to continue. Checkpoints are saved every SAVE_FREQ steps and backed up every 15 minutes, so a disconnect costs at most a few minutes of training.

In [12]:
def train(exp, script):
    cfg = OUT / "train" / exp / "checkpoints" / "last" / "pretrained_model" / "train_config.json"
    if cfg.exists():
        print(f"Resuming {exp} from {cfg.parent.parent.resolve().name}")
        cmd = f"lerobot-train --config_path={cfg} --resume=true"
    else:
        if (OUT / "train" / exp).exists():   # a crashed run with no usable checkpoint
            print(f"Removing incomplete run {exp}"); shutil.rmtree(OUT / "train" / exp)
        cmd = f"bash scripts/{script}"
    result = run(cmd, log=f"train_{exp}.log", show=IMPORTANT, env=COMMON, hours=TRAIN_HOURS)
    run(sync_cmd("push"))   # back up right away
    done = (OUT / "train" / exp / "checkpoints" / "last").resolve().name
    print(f"{exp}: latest checkpoint = step {done}" + (" (time budget hit - run again to continue)" if result == "timeout" else ""))

if want("train_A"):
    train("expert_only", "02_train_expert_only.sh")
if want("train_B"):
    train("expert_plus_vlm", "03_train_expert_plus_vlm.sh")

Resuming expert_plus_vlm from 016000
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
Traceback (most recent call last):
OSError: libnppicc.so.12: cannot open shared object file: No such file or directory
Traceback (most recent call last):
    raise OSError(f"Could not load this library: {path}") from e
OSError: Could not load this library: /content/venv/lib/python3.12/site-packages/torchcodec/libtorchcodec_core8.so
Traceback (most recent call last):
OSError: libnppicc.so.12: cannot open shared object file: No such file or directory
Traceback (most recent call last):
    raise OSError(f"Could not load this library: {path}") from e
OSError: Could not load this library: /content/venv/lib/python3.12/site-packages/torchcodec/libtorchcodec_core7.so
Traceback (most recent call last):
OSError: libnppicc.so.12: cannot open shared object file: No such file or directory
Traceback (most re

### eval_trained: closed-loop evaluation of the trained checkpoints

In [13]:
if want("eval_trained"):
    final = lambda exp: (OUT / "train" / exp / "checkpoints" / "last").resolve().name
    for exp in ("expert_only", "expert_plus_vlm"):
        if (OUT / "train" / exp).exists():
            print(f"{exp}: evaluating step {final(exp)} (should equal STEPS={STEPS:06d} for a finished run)")
    run("bash scripts/04_eval_trained.sh", log="eval_trained.log", show=IMPORTANT, env=COMMON)
    show_video("01_expert_only")

### quant_list / quant: quantization

Run `quant_list` first. Read which module-name prefixes belong to the VLM backbone and which to the action expert, set `VLM_RE` and `EXPERT_RE` in CONFIG so each matches only its part, then run `quant`.

In [14]:
CKPT_A = "outputs/train/expert_only/checkpoints/last/pretrained_model"
if want("quant_list"):
    run(f"python -m vla_study.quantize --list {CKPT_A}")
if want("quant"):
    run("bash scripts/05_quant_sweep.sh", log="quant.log", show=IMPORTANT,
        env={**COMMON, "VLM_RE": VLM_RE, "EXPERT_RE": EXPERT_RE})
    print(json.dumps(json.load(open(OUT / "offline_bench.json")), indent=1)[:3000])

### paraphrase_generate: build the instruction sets, then REVIEW them

In [15]:
TASKS = OUT / "data" / f"tasks_{SUITE}.json"
PARA  = OUT / "data" / f"paraphrases_{SUITE}.json"
if want("paraphrase_generate"):
    run(f"python -m vla_study.paraphrase extract --suite {SUITE} --out {TASKS}")
    run(f"python -m vla_study.paraphrase generate --tasks {TASKS} --out {PARA}" + (f" --llm {LLM}" if LLM else ""))
if PARA.exists():
    import pandas as pd
    sets = json.load(open(PARA))
    pd.set_option("display.max_colwidth", 80)
    display(pd.DataFrame({lvl: list(m.values()) for lvl, m in sets.items() if lvl != "original"},
                         index=list(sets["original"])))

**Review every row above.** If a paraphrase changes the meaning (different object, different place), fix it in FIXES below and run the cell. Leave FIXES empty if everything is fine. The swapped column is meant to be wrong: it's the control.

In [16]:
FIXES = {
    # "level": {"original instruction": "corrected paraphrase"},
    # "llm": {"pick up the alphabet soup and place it in the basket": "put the alphabet soup into the basket"},
}
if PARA.exists() and FIXES:
    sets = json.load(open(PARA))
    for lvl, fixes in FIXES.items():
        for orig, new in fixes.items():
            assert orig in sets[lvl], f"not an original instruction: {orig!r}"
            sets[lvl][orig] = new
    json.dump(sets, open(PARA, "w"), indent=2)
    print("saved fixes; the backup loop will upload them")

### paraphrase_eval: robustness evaluation


In [17]:
if want("paraphrase_eval"):
    assert PARA.exists(), "run paraphrase_generate (and review) first"
    run("bash scripts/06_paraphrase_sweep.sh", log="paraphrase.log", show=IMPORTANT, env={**COMMON, "LLM": LLM})

### results: table, chart and GIFs

In [18]:
if want("results"):
    from IPython.display import Image, Markdown
    run("bash scripts/07_results.sh", log="results.log")
    display(Markdown(open(OUT / "results" / "results.md").read()))
    display(Image(str(OUT / "results" / "success_rates.png")))

## Finish: final backup (always runs)

In [19]:
sync_proc.terminate()
run(sync_cmd("prune", "--keep 1"))
run(sync_cmd("push"))
print(f"All backed up to https://huggingface.co/{HF_RUNS_REPO}")

[sync] pruned /content/vla-outputs/train/expert_plus_vlm/checkpoints/018000
[sync] pruned /content/vla-outputs/train/expert_plus_vlm/checkpoints/019000
No files have been modified since last commit. Skipping to prevent empty commit.
[sync] push done 18:14:03
All backed up to https://huggingface.co/arckive71/vla-study-runs
